# Plots maps carbon densities differences

In [ ]:
# Libraries
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir_nearth = '../data/naturalearth/'
dir_zen = '../paper_ecosystem_carbon/output/zenodo/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'

---

### Pre-processing

In [ ]:
# Get carbon density data
ds_cspatial = xr.open_dataset(f'{dir_zen}ecosystem_carbon_cspatial_v1-0.nc')
ds_cdgvm = xr.open_dataset(f'{dir_zen}ecosystem_carbon_cdgvm_v1-0.nc')

# Get coastline data
coastline110 = gpd.read_file(
    f'{dir_nearth}ne_110m_coastline/ne_110m_coastline.shp')

In [ ]:
# List of etype entries in order to be plotted
list_etype = ['trop_ef', 'trop_df', 'temp_dbf', 'temp_dnf', 'temp_ebf', 
              'temp_enf', 'glob_shrub', 'glob_grass', 'glob_tundr', 'glob_mount']

---

### Plot function

In [ ]:
def plot_maps_per(ctype, cmap):
        
    # Plot
    fig, axs = plt.subplots(figsize=(12, 14), ncols=2, nrows=5, dpi=300)
    axs = axs.ravel()

    plotted_data = []

    # Loop through every ecosystem type
    for i in range(0, len(list_etype)):
        v = list_etype[i]
        if ctype == 'cveg':
            vmax = 100
        if ctype == 'csoil':
            vmax = 100
            
        coastline110.plot(ax=axs[i], color='#000000', linewidth=0.5) 

        ds_cspatial.land_sea_mask.plot.imshow(
                    ax=axs[i], 
                    cmap='Greys', vmax=2, add_colorbar=False, alpha=0.5)


        da_cspatial = ds_cspatial[f'{ctype}_{v}']
        da_cdgvm = ds_cdgvm[f'{ctype}_{v}']
        im = (((da_cspatial / da_cdgvm) -1) * 100).plot.imshow(
            ax = axs[i], 
            vmin=-vmax, vmax=vmax, 
            cmap=cmap, add_colorbar=False)

        plotted_data.append(im)
        
        axs[i].set_title(da_cspatial.attrs['ecosystem_type'], size='x-large')
        axs[i].axis('off')
        axs[i].set_ylim(-60, 85)
    

    # Colorbars
    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.01])
    if ctype == 'cveg':
        legend_label_str = 'vegetation carbon'
    if ctype == 'csoil':   
        legend_label_str = r'soil organic carbon'

    legend_label = (f'Difference in {legend_label_str} '
                    f'between cSpatial and cDGVM (%)')
    
    cbar = fig.colorbar(plotted_data[0], 
                        cax=cbar_ax, orientation='horizontal', 
                        label=legend_label, 
                        extend='both', pad=0.1, aspect=45, shrink=0.8)

    cbar_ax = cbar.ax
    for i, ii, iii in zip([-100, 100],
                          ['cDGVM higher', 'cSpatial higher'],
                          ['left', 'right']):
        cbar_ax.text(i, 1.8, ii, ha=iii, va='center', c='#000000')

    # Layout
    plt.tight_layout(rect=[0, 0.07, 1, 1])

    # Export
    if ctype == 'cveg':
        fstr_out = 'fig_s03_cveg_reldiff_cspatial_cdgvm'
    if ctype == 'csoil':
        fstr_out = 'fig_s05_csoil_reldiff_cspatial_cdgvm'

    plt.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300, 
            bbox_inches='tight', pad_inches=0.1); 
    plt.savefig(f'{dir08}png/{fstr_out}.png', dpi=300, 
            bbox_inches='tight', pad_inches=0.1);

---

### Plots

In [ ]:
plot_maps_per('cveg', 'RdBu_r')

In [ ]:
plot_maps_per('csoil', 'RdBu_r')